# Correlation Leakage - Nov 3

## Setup
* LFR benchmark graphs with `1,000` nodes 
* average degree = `{15, 20, 25}`
* maximum degree = `50`
* `20` equally spaced values for Mixing parameter $\mu \in$ `[0, 1]`
* `20` different networks for the same LFR parameters 
* 10 independent trials for each method 
* Picking the `best` of the `10` trials based on `modularity`, `description length`, `NMI`, etc. 

# TODO (Nov 7)
* Plot clusterings from individual runs - read from disk

-----

Notes (nov 2)
* Use Santo's Robustness modularity framework to identify meaningful clusters for higher mu
* Markov Diff is different from CL Diff - number of clusters differs with higher mu
* MC performance does not monotonically decrease with mu 
* Explore Hierarchical networks? Brain data?
* Generating dendrograms from CL and MC
* Ground truth of dendrograms - tree edit distance?
* Reconstructing dendrogram from all tau values 
* Compare the clusterings with the different levels of the ground truth dendrogram - find the best level of the tree 

In [108]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
# ^^^ pyforest auto-imports - don't write above this line
%load_ext autoreload
%autoreload 2

Note: to be able to use all overlapping methods, you need to install some additional packages:  {'ASLPAw'}
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [109]:
! python -m pip install clusim

In [110]:
import random
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
from networkx.algorithms.community import modularity
import networkx as nx
import scipy
import numpy as np
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff

import sys
sys.path.extend(['./', '../'])

from src.clustering import LFRGraph, CorrelationClustering
from src.utils import Paths

In [111]:
%matplotlib inline

# plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 20
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['axes.titlesize'] = 20

pd.options.plotting.backend = 'plotly'
pio.templates.default = 'presentation+plotly_dark'
pio.renderers.default = 'iframe'

sns.set_context('talk')
sns.set_style("ticks")

SMALL_SIZE = 30
MEDIUM_SIZE = 30
BIGGER_SIZE = 30

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=20)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [139]:
def make_plot(df, suptitle):
    rows, cols = 2, 3
    metrics = ['clusters', '', 'modularity', 'NMI', 'AMI', 'EC']

    colors = px.colors.qualitative.Plotly
    fill_alpha = 0.35
    fill_colors = [(c[: c.rfind(')')] + f', {fill_alpha})').replace('rgb', 'rgba')  for c in colors]
    
    d = {'clusters': 'Number of Clusters', 'NMI': 'Normalized Mutual Information', 'modularity': 'Modularity',
         'AMI': 'Adjusted Mutual Information', 'EC': 'Element-centric Score'}
    subplot_titles = [d.get(metric, metric) for metric in metrics]
    
    m = [metric for metric in metrics if metric != '']
    df = (
        df
        .fillna(0)
        .rename(columns={'num_clusters': 'clusters'})
        [['method', 'mu'] + m]
        .pivot_table(
            index=['method', 'mu'], 
            aggfunc=['mean', 'sem'],
            fill_value=0,
        )
        .reset_index()
    )

    df.columns = df.columns.to_flat_index().str.join('_')
    df = df.rename(columns={'mu_': 'mu', 'method_': 'method'})
    
    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=subplot_titles,
        shared_xaxes=True, vertical_spacing=0.12
    )
    
    # legend_titles = {'CO-C': 'Consensus-C', 'RW-C': 'Random Walk-C'}
    legend_titles = {}
    
    for i, metric in enumerate(metrics):
        row = i // cols + 1
        col = i % cols + 1
        if row == 1 and col == 2:
            continue 
            
        for j, (method, method_grp) in enumerate(df.groupby('method')):
            showlegend = i == 0  # show legend entries for the first plot
            color = colors[j]
            
            x = method_grp.mu.values.tolist()
            y = method_grp[f'mean_{metric}'].tolist()
            y_upper = (method_grp[f'mean_{metric}'] + method_grp[f'sem_{metric}']).tolist()
            y_lower = (method_grp[f'mean_{metric}'] - method_grp[f'sem_{metric}']).tolist()
            
            # error bar
            fig.add_trace(
                go.Scatter(
                    x=x+x[::-1], # x, then x reversed
                    y=y_upper+y_lower[::-1], # upper, then lower reversed
                    fill='toself',
                    fillcolor=fill_colors[j],
                    line=dict(color='rgba(255,255,255,0)', width=0),
                    hoverinfo="skip",
                    showlegend=False,
                    legendgroup=method,
                ),
                row=row, col=col
            )
            
            # main plot - mean
            fig.add_trace(
                go.Scatter(
                    x=x, y=method_grp[f'mean_{metric}'], 
                    name='', text=method,
                    mode='lines+markers', 
                    opacity=0.7, 
                    line=dict(color=color),
                    marker=dict(line_color='DarkSlateGrey', line_width=2),
                    showlegend=showlegend,
                    legendgroup=method,  # this can be any string, not just "group"
                    legendgrouptitle_text=legend_titles.get(method, method),
                ), 
                row=row, col=col
            )
            
        fig.update_xaxes(title_font=dict(size=18), title=r'$\mu$', row=row, col=col)
        fig.update_yaxes(rangemode="tozero", title=metric, row=row, col=col)
        if i != 0:  # update range for everything except the first plot
            fig.update_yaxes(range=(-0.05, 1.05), row=row, col=col)
    
    legend_y = 0.9 if df.method.nunique() < 6 else 1.1
    
    fig.update_layout(title=suptitle, height=900, 
                      legend=dict(
                          orientation="v",
                          # entrywidth=75,
                          yanchor="top",
                          y=legend_y,
                          font=dict(size=10),
                          xanchor="left",
                          x=0.45)
                     )
    return fig

In [140]:
def varying_tau_plot(metric, methods):
    colors = {method: px.colors.qualitative.Plotly[i] for i, method in enumerate(methods)}
    fill_alpha = 0.35
    fill_colors = {method: (c[: c.rfind(')')] + f', {fill_alpha})').replace('rgb', 'rgba')  for method, c in colors.items()}
    fig = go.Figure()

    df = (
        pd.concat([cl_results_df, mc_results_df], ignore_index=True)
        .rename(columns={'num_clusters': 'clusters'})
        .query('method.isin(@methods)')
    )

    # Add traces, multiple plots for each slider step
    for tau_ix in range(df.tau_ix.max() + 1):
        for i, method in enumerate(methods):  # for each method
            filt_df = (
                df
                .query('tau_ix==@tau_ix & method==@method', engine='python')
            )

            if len(filt_df) == 0:
                fig.add_trace(go.Scatter(name=f'{method}_{tau_ix}_blank', visible=False))
                fig.add_trace(go.Scatter(name=f'{method}_{tau_ix}_blank_error', visible=False))
            else:
                df_ = (
                    filt_df
                    .pivot_table(
                    index=['method', 'mu'], 
                    aggfunc=['mean', 'sem'],
                    fill_value=0)
                    .reset_index()
                )
                df_.columns = df_.columns.to_flat_index().str.join('_') 
                df_ = df_.rename(columns={'mu_': 'mu', 'method_': 'method'})

                x = df_.mu.values.tolist()
                y = df_[f'mean_{metric}'].tolist()
                y_upper = (df_[f'mean_{metric}'] + df_[f'sem_{metric}']).tolist()
                y_lower = (df_[f'mean_{metric}'] - df_[f'sem_{metric}']).tolist()

                # error bar
                fig.add_trace(
                    go.Scatter(
                        x=x+x[::-1], # x, then x reversed
                        y=y_upper+y_lower[::-1], # upper, then lower reversed
                        fill='toself',
                        fillcolor=fill_colors[method],
                        line=dict(color='rgba(255,255,255,0)', width=0),
                        hoverinfo="skip",
                        showlegend=False,
                        visible=False,
                        legendgroup=method,
                    )
                )

                # actual data 
                fig.add_trace(
                    go.Scatter(
                        visible=False,
                        marker_color=colors[method],
                        legendgroup=method,
                        name=method,
                        x=df_['mu'],
                        y=df_[f'mean_{metric}'],
                        mode='markers+lines',
                        opacity=0.7)
                )


    # Make 0th trace visible
    for i in range(2*len(methods)):
        fig.data[i].visible = True

    # Create and add slider
    steps = []
    # title_strs = ['tau = i' for i in ]
    step_labels = [''] * df.tau_ix.nunique()
    step_labels[0] = '0'
    step_labels[4] = 'max'
    step_labels[-1] = 'diff'

    for i in range(0, len(fig.data), 2*len(methods)):  # skip over lines for the same method
        i_ = i // (2 * len(methods))
        step = dict(
            method="update",
            label=step_labels[i_],
            args=[{"visible": [False] * len(fig.data)},
                  # {"title": f'{step_labels[i_]}'},
                 ],  # layout attribute
        )
        for j in range(0, 2*len(methods), 2):  # turn on i and i + j 
            step["args"][0]["visible"][i+j] = True  # Toggle i'th trace to "visible"
            step["args"][0]["visible"][i+j+1] = True  # Toggle i'th trace to "visible"
        steps.append(step)


    sliders = [dict(
        active=0,
        name='ix',
        currentvalue={"prefix": r"Tau: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        title=f'Variation of {metric!r} over taus for {", ".join(list(methods))!r}',
        sliders=sliders
    )
    fig.update_xaxes(title=r'$\mu$')

    # y_range = (0, df.clusters.max()) if metric == 'clusters' else (0, 1)
    y_range = None if metric == 'clusters' else (-0.05, 1.05)
    fig.update_yaxes(range=y_range, title=metric)

    return fig

In [122]:
def _make_plot(df, suptitle, cmap='tab10'):
    hue_order = sorted(df.method.unique())
    ncol = len(hue_order) // 9 + 1
    if ncol > 1:
        figsize = (25, 10)
    else:
        figsize = (15, 10)
    fig, axs = plt.subplots(figsize=figsize, ncols=2, nrows=2, sharex=True, layout='constrained')

    axs = axs.flatten()

    pal = sns.color_palette(cmap, len(hue_order))
    for i, (ax, y) in enumerate(zip(axs, ['num_clusters', 'modularity', 'NMI', 'EC'])):
        sns.lineplot(data=df, x='mu', y=y, hue='method', hue_order=hue_order, marker='o', ax=ax, palette=pal, alpha=0.75);

        if i == 0:  # add baseline for before and num_clusters
            sns.lineplot(data=df, x='mu', y='orig_clusters', marker='o', color='black', ax=ax)

        ax.grid();
        # Hide the right and top spines
        ax.spines.right.set_visible(False)
        ax.spines.top.set_visible(False)

        # Only show ticks on the left and bottom spines
        ax.yaxis.set_ticks_position('left');
        ax.xaxis.set_ticks_position('bottom');

        if i == 1:
            # get the legend object
            leg = ax.legend(bbox_to_anchor=(1.04, 0.25), loc="lower left", borderaxespad=0, ncol=ncol)
            # change the line width for the legend
            for line in leg.get_lines():
                line.set_linewidth(6.0)

        else:
            ax.legend().set_visible(False)

    fig.suptitle(suptitle);
    return 

# data prep

In [123]:
paths = Paths()
cl_results_path = paths.results_dir / 'stats_CL.csv'
mc_results_path = paths.results_dir / 'stats_MC.csv'
other_results_path = paths.results_dir / 'stats_others.csv'

In [124]:
cl_results_df = pd.read_csv(cl_results_path, on_bad_lines='skip')

cl_results_df = (
    cl_results_df
    .assign(method=lambda df_: df_.method.str.replace('-exact', '', regex=False))
)
mask = ~((cl_results_df.method=='GW-C') & (cl_results_df.tau_ix>4))  # delete rows with tau > tau_max for GW 
cl_results_df = cl_results_df[mask]
cl_results_df.drop_duplicates(subset=cl_results_df.columns[1: ], inplace=True, keep='last')



display(len(cl_results_df), f'{cl_results_df.method.unique().tolist()}', cl_results_df.tail(2))

109791

"['GW-C', 'RW-D', 'RW-C', 'Diff-C', 'GW-D', 'CO-C', 'Diff-C-ER', 'RW-C-ER', 'CO-C-ER', 'RW-D-ER']"

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,running_time,metric,score,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
119869,Thu Nov 10 23:27:24 2022,SBM-big-small-0.1,NaN,NaN,NaN,RW-C,0.8597,3.467957,7,2.815894,4,0.1255,leakage_score,21.153674,5,0.2956,0.6507,0.6412,0.4930,/N/slate/ssikdar/data-dump/corr-cluster/result...
119870,Thu Nov 10 23:27:25 2022,SBM-big-small-0.1,NaN,NaN,NaN,RW-C,0.8597,3.467957,8,3.467957,4,0.1706,leakage_score,14.286019,66,0.1705,0.4897,0.3260,0.3212,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [125]:
mc_results_df = pd.read_csv(mc_results_path)
mc_results_df.drop_duplicates(subset=mc_results_df.columns[1: ], inplace=True, keep='last')

display(len(mc_results_df), mc_results_df.method.unique(), mc_results_df.tail(2))

17412

array(['MC-Diff-C', 'MC-RW-C'], dtype=object)

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,metric,score,running_time,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
17410,Thu Nov 10 23:27:38 2022,SBM-big-small-0.1,NaN,NaN,NaN,MC-Diff-C,0.009,0.054086,7,0.042804,4,0.1035,leakage_score,0.280358,4,0.3241,0.8,0.7964,0.625,/N/slate/ssikdar/data-dump/corr-cluster/result...
17411,Thu Nov 10 23:27:39 2022,SBM-big-small-0.1,NaN,NaN,NaN,MC-Diff-C,0.009,0.054086,8,0.054086,4,0.1231,leakage_score,0.224743,4,0.3241,0.8,0.7964,0.625,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [126]:
other_results_df = pd.read_csv(other_results_path)
other_results_df.drop_duplicates(subset=other_results_df.columns[1: ], inplace=True, keep='last')

other_results_df = (
    other_results_df
    .query('method!="method"')
    .convert_dtypes(convert_floating=True, convert_integer=True)
    .astype({'num_clusters': 'int'})
    .assign(method=lambda df_: df_.method.replace({'greedy_modularity': 'CNM',
                                                   'label_propagation': 'Label Prop',
                                                   'infomap': 'Infomap',
                                                   'eigenvector': 'eig',
                                                   'sbm_dl': 'SBM',
                                                   'sbm_dl_nested': 'Nested SBM'}))
)

display(len(other_results_df), f'{other_results_df.method.unique().tolist()}', other_results_df.tail(2))

9445

"['leiden', 'Label Prop', 'Infomap', 'louvain', 'CNM', 'SBM', 'Nested SBM']"

,timestamp,name,mu,avg_k,max_k,method,orig_clusters,metric,score,running_time,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
9443,Fri Nov 11 00:44:28 2022,SBM-easy-0.1,<NA>,<NA>,<NA>,SBM,4,nmi,1.0,0.3407,4,0.5006,1.0,1.0,1.0,/N/slate/ssikdar/data-dump/corr-cluster/result...
9444,Fri Nov 11 00:44:31 2022,SBM-easy-0.1,<NA>,<NA>,<NA>,Nested SBM,4,nmi,1.0,1.7791,4,0.5006,1.0,1.0,1.0,/N/slate/ssikdar/data-dump/corr-cluster/result...


# Plots

## Results for all Correlation Leakage

In [127]:
make_plot(df=cl_results_df, suptitle='Correlation Leakage')

### CL reproducing Louvain when $\tau = 0$ for Diffusion and Graph Walk

In [142]:
# Pick the methods that reduce to modularity - Diff-C, GW-C, MC-RW-C?
keep = {"Diff-C", "GW-C"}
df = pd.concat([
    cl_results_df
    .query('method.isin(@keep) & tau_ix==0', engine='python'),
    other_results_df
    .query('method=="louvain"')
], ignore_index=True)

make_plot(df=df, suptitle='Reproducing Louvain')

ValueError: 
    Invalid value of type 'builtins.str' received for the 'fillcolor' property of scatter
        Received value: '#636EF, 0.35)'

    The 'fillcolor' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color:
            aliceblue, antiquewhite, aqua, aquamarine, azure,
            beige, bisque, black, blanchedalmond, blue,
            blueviolet, brown, burlywood, cadetblue,
            chartreuse, chocolate, coral, cornflowerblue,
            cornsilk, crimson, cyan, darkblue, darkcyan,
            darkgoldenrod, darkgray, darkgrey, darkgreen,
            darkkhaki, darkmagenta, darkolivegreen, darkorange,
            darkorchid, darkred, darksalmon, darkseagreen,
            darkslateblue, darkslategray, darkslategrey,
            darkturquoise, darkviolet, deeppink,

### performance variation for different values of $\tau$

In [129]:
metric = 'clusters'
methods = {'CO-C', 'GW-C', 'RW-C', 'Diff-C'}
# methods = {'CO-C', 'GW-C', 'RW-C', 'Diff-C', 'MC-RW-C', 'MC-Diff-C'}
# metric = 'modularity'
# metric = 'EC'
# metric = 'NMI'
# metric = 'AMI'
metric = 'clusters'
varying_tau_plot(metric=metric, methods=methods)

In [130]:
# methods = {'CO-C', 'GW-C', 'RW-C', 'Diff-C', 'MC-RW-C', 'MC-Diff-C'}
methods = {'MC-Diff-C'}

# metric = 'clusters'
# metric = 'modularity'
metric = 'EC'
# metric = 'NMI'
# metric = 'AMI'

varying_tau_plot(metric=metric, methods=methods)

In [131]:
df = (
    cl_results_df
    .query('tau_ix<=4')
)

make_plot(df, suptitle='CL upto tau max')

## Markov Stability

In [132]:
make_plot(df=mc_results_df, suptitle='Markov Stability (all)')

### Markov Clustering when $\tau \approx 1$

In [133]:
var = 1

df = (
    mc_results_df
    .assign(diff=lambda df_: abs(df_.tau - var))
    .sort_values(by='diff')
    .drop_duplicates(subset=['name', 'mu', 'avg_k', 'method'])
)

make_plot(df, suptitle=f'Markov Clustering when tau = {var}')

In [134]:
avg_k = 20
df = (
    mc_results_df
    .query('tau_ix==4')
    .query('avg_k==@avg_k')
)

make_plot(df, suptitle=f'Markov Clustering when tau = tau max avg_k = {avg_k}')

### Comparing the "best" of Markov with CL

In [135]:
tau_ix = 8

df = pd.concat([
        mc_results_df
        .query('tau_ix==@tau_ix'),
        cl_results_df
        .query('tau_ix==@tau_ix')
    ], ignore_index=True
)

make_plot(df, suptitle=f'Markov Clustering v. CL when tau = tau max')

## other methods

In [136]:
make_plot(df=other_results_df, suptitle='Other Methods')

### other methods v. CL (all)

In [137]:
keep = {'louvain', 'SBM', 'Nested SBM', 'Infomap'}

df = pd.concat([
    cl_results_df
    .query('tau_ix==4')
    ,
    other_results_df
    .query('method.isin(@keep)')
], ignore_index=True)

make_plot(df=df, suptitle='Other methods vs. CL (all taus)')

In [141]:
avg_k = 15
keep = {'louvain', 'SBM', 'Nested SBM', 'Infomap', 'MC-GW-C'}

df = pd.concat([
    cl_results_df
    .query('tau_ix==0')
    .query('avg_k==@avg_k'),
    other_results_df
    .query('method.isin(@keep)'),
    mc_results_df
    .query('method.isin(@keep)')
], ignore_index=True)

make_plot(df, suptitle=f'Results when avg deg = {avg_k} and tau = 0')

ValueError: 
    Invalid value of type 'builtins.str' received for the 'fillcolor' property of scatter
        Received value: '#636EF, 0.35)'

    The 'fillcolor' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color:
            aliceblue, antiquewhite, aqua, aquamarine, azure,
            beige, bisque, black, blanchedalmond, blue,
            blueviolet, brown, burlywood, cadetblue,
            chartreuse, chocolate, coral, cornflowerblue,
            cornsilk, crimson, cyan, darkblue, darkcyan,
            darkgoldenrod, darkgray, darkgrey, darkgreen,
            darkkhaki, darkmagenta, darkolivegreen, darkorange,
            darkorchid, darkred, darksalmon, darkseagreen,
            darkslateblue, darkslategray, darkslategrey,
            darkturquoise, darkviolet, deeppink,

## Plot clusterings

In [107]:
def read_file_into_dict(path):
    d = {}
    with open(path) as reader:
        for line in reader:
            x, y = map(int, line.split())
            d[x] = y
    return d

In [40]:
def get_edge_trace(G, edge_width, inter_edge_color, intra_edge_color, comm_labels):
    if comm_labels is None:
        comm_labels = {}
        
    inter_edge_width = edge_width  
    intra_edge_width = edge_width * 1.5

    intra_edge_x, intra_edge_y = [], []
    inter_edge_x, inter_edge_y = [], []
    
    for u, v in G.edges():
        x0, y0 = G.nodes[u]['pos']
        x1, y1 = G.nodes[v]['pos']
        
        if comm_labels.get(u, u) == comm_labels.get(v, v):
            intra_edge_x.extend([x0, x1, None])
            intra_edge_y.extend([y0, y1, None])
        else:
            inter_edge_x.extend([x0, x1, None])
            inter_edge_y.extend([y0, y1, None])

    intra_edge_trace = go.Scattergl(
        x=intra_edge_x, y=intra_edge_y,
        line=dict(width=intra_edge_width, color=intra_edge_color),
        hoverinfo='skip',
        opacity=0.8,
        mode='lines')
            
    inter_edge_trace = go.Scattergl(
        x=inter_edge_x, y=inter_edge_y,
        line=dict(width=inter_edge_width, color=inter_edge_color),
        hoverinfo='skip',
        opacity=0.8,
        mode='lines')
    return intra_edge_trace, inter_edge_trace

def get_node_trace(G, node_size, marker_width):
    node_x, node_y = [], []

    for node in G.nodes():
        x, y = G.nodes[node]['pos']
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scattergl(
        name='nodes',
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=[],
            size=node_size,
            line=dict(
                width=marker_width,
                color='black',
            ),
            line_width=2)
    )

    return node_trace


def update_node_colors_and_texts(G, comm_labels):
    """
    Return updated node trace of node colors and texts based on comm_labels 
    """
    node_colors, node_text = [], []
    if comm_labels is not None:
        colors = px.colors.qualitative.Pastel 
        for node in G.nodes():
            comm_label = comm_labels[node]
            node_colors.append(colors[comm_label])
            node_text.append(f'Node: {node}<br>Cluster: {comm_label}')
    else:
        node_colors = ['orange'] * G.order()
        node_text = [f'Node: {node}' for node in G.nodes()]
        
    # node_trace.marker.color = node_colors
    # node_trace.text = node_text
    return node_colors, node_text


def plot_graph(G, comm_labels=None, layout=None, node_size=10, edge_width=0.25, edge_color='gray', marker_width=10, fig=None, return_traces=False):
    if G.order() < 50:
        node_size = 20
        edge_width = 1.5
        edge_color = 'silver'
        
    if fig is None:
        fig = go.Figure()
    
    if layout is not None:
        layout_alg = getattr(nx.layout, f'{layout}_layout')
        nx.set_node_attributes(G, name='pos', values=layout_alg(G))
    
    intra_edge_trace, inter_edge_trace = get_edge_trace(G, edge_width=edge_width, inter_edge_color=edge_color, intra_edge_color='silver',
                                                        comm_labels=comm_labels)
    node_trace = get_node_trace(G, marker_width=marker_width, node_size=node_size)
    
    node_colors, node_text = update_node_colors_and_texts(G=G, comm_labels=comm_labels)
    node_trace.marker.color = node_colors
    node_trace.text = node_text
    
    traces =  intra_edge_trace, inter_edge_trace, node_trace  # nodes drawn at the end
    fig.add_traces(traces)
    
    fig.update_layout(showlegend=False, 
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
    if return_traces:
        return traces
    else:
        return fig
    

In [12]:
def make_community_heatmap(G, comm_labels):
    nx.set_node_attributes(G, name='cluster', values=comm_labels)
    mat = nx.attribute_mixing_matrix(G, attribute='cluster')
    # iu = np.triu_indices(mat.shape[0], 1)
    # mat[iu] = None
    text = np.round(mat, 3)
    
    clu_text = [f'{i}' for i in range(mat.shape[0])]
    heatmap = go.Heatmap(x=clu_text, y=clu_text, z=mat, text=text, texttemplate='%{text}',
                         showscale=False, hoverongaps=False,
                        )
    return heatmap

In [ ]:
# G = nx.random_geometric_graph(100, 0.35)
# comm_labels = {n: n % 3 for n in G.nodes}
G = nx.planted_partition_graph(4, 32, 0.4, 0.05)
comm_labels = nx.get_node_attributes(G, name='block')

fig = go.Figure()

fig = make_subplots(
        rows=1, cols=2, subplot_titles=['Network', 'Heatmap'],
        column_widths=[0.7, 0.3],
        vertical_spacing=0.0,
    )


traces = plot_graph(G, comm_labels=comm_labels, node_size=15, edge_width=0.5, layout='spring')
fig.add_traces(traces, rows=1, cols=1)
# fig.add_trace(node_trace, row=1, col=1)

fig.add_trace(make_community_heatmap(G, comm_labels=comm_labels), row=1, col=2)
fig.update_yaxes(autorange='reversed', row=1, col=2)

fig.update_layout(showlegend=False, 
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  height=700)


fig

# Widgets

## TODO (Nov 10):
* Viz one network, read clustering, edge distributions
* Start with just the network and just update the colors of the nodes wh
* Viz two clusterings of the same network - fix the node positions - different methods - Ground truth as a separate method. have menus for different methods for CL / tau indices 

In [13]:
from ipywidgets import widgets

## graph viz

In [19]:
sizes = [50, 50, 50, 50]
mu = 0.1
p_in, p_out = 1 - mu, mu

G = nx.random_partition_graph(sizes, p_in=p_in, p_out=p_out, seed=1)
comm_labels = nx.get_node_attributes(G, name='block')
layout = 'spring'
nx.set_node_attributes(G, name='pos', values=getattr(nx.layout, f'{layout}_layout')(G))

print(G)

Graph named 'stochastic_block_model' with 200 nodes and 5855 edges


### testing out general things

In [52]:
# test out with 3 different clusterings of the same network - SBM of 200 nodes 
fig = plot_graph(G, comm_labels=None)
fig.update_layout(width=500, height=500)
fig

In [67]:
# use update_traces to modify node colors and text - callback event 
node_colors, node_texts = update_node_colors_and_texts(G=G, comm_labels=comm_labels)
fig.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors), text=node_texts)
fig

### back to widgets

In [20]:
name = 'toy'
methods = ['ground-truth', 'louvain', 'leiden']

op_cl = [f'./test/{method}.txt' for method in methods]
test_df = pd.DataFrame(dict(name=[name]*3, method=['ground-truth', 'louvain', 'leiden'], output_cluster_path=op_cl))
test_df.head()

,name,method,output_cluster_path
0,toy,ground-truth,./test/ground-truth.txt
1,toy,louvain,./test/louvain.txt
2,toy,leiden,./test/leiden.txt


In [21]:
G = nx.barbell_graph(4, 3)
layout = 'spring'
nx.set_node_attributes(G, name='pos', values=getattr(nx.layout, f'{layout}_layout')(G))

print(G)

Graph with 11 nodes and 16 edges


In [41]:
method_box = widgets.Dropdown(
    description='Algorithm:   ',
    value='ground-truth',
    options=test_df.method.unique().tolist(),
)

container = widgets.HBox([method_box])
traces = plot_graph(G, return_traces=True)


g = go.FigureWidget(data=traces,
                    layout=go.Layout(
                        title=dict(text='Clustering visualization'),
                        width=700, height=700,
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        showlegend=False
                    ))

In [42]:
def validate():
    if method_box.value in test_df['method'].unique():
        return True
    else:
        return False


def response(change):
    if validate():
        method = method_box.value
        temp_df = test_df[test_df.method==method]
        # read the clustering, update the node traces 
        cluster_path = temp_df.output_cluster_path.values[0]
        comm_labels = read_file_into_dict(cluster_path)
        num_clusters = len(set(comm_labels.values()))
        
        node_colors, node_texts = update_node_colors_and_texts(G=G, comm_labels=comm_labels)
        # fig.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors), text=node_texts)
        
        with g.batch_update():
            g.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors), text=node_texts)
            g.layout.title = f'{G.name}    |V| = {G.order():,}   |E| = {G.size():,}<br>{method} {num_clusters:,} clusters'
            # g.layout.yaxis.title = 'Number of Delays'


method_box.observe(response, names="value")

In [43]:
widgets.VBox([method_box, g])

## second panel

In [106]:
def validate_both():
    acceptable_methods = set(test_df['method'].unique())
    if method_box_left.value in acceptable_methods and method_box_right.value in acceptable_methods:
        return True
    else:
        return False


def response_both(change):
    if validate_both():
        method_left = method_box_left.value
        temp_df_left = test_df[test_df.method==method_left]
        comm_labels_left = read_file_into_dict(temp_df_left.output_cluster_path.values[0])
        num_clusters_left = len(set(comm_labels_left.values()))
        node_colors_left, node_texts_left = update_node_colors_and_texts(G=G, comm_labels=comm_labels_left)
        
        method_right = method_box_right.value
        temp_df_right = test_df[test_df.method==method_right]
        comm_labels_right = read_file_into_dict(temp_df_right.output_cluster_path.values[0])
        num_clusters_right = len(set(comm_labels_right.values()))
        node_colors_right, node_texts_right = update_node_colors_and_texts(G=G, comm_labels=comm_labels_right)
        
        with g.batch_update():
            g.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors_left), text=node_texts_left, row=1, col=1)
            g.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors_right), text=node_texts_right, row=1, col=2)
            
            g.layout.annotations[0].update(text=f'{method_left} {num_clusters_left:,} clusters')
            g.layout.annotations[1].update(text=f'{method_right} {num_clusters_right:,} clusters')
        
        return

In [101]:
method_box_left = widgets.Dropdown(
    description='Algorithm:   ',
    value='ground-truth',
    options=test_df.method.unique().tolist(),
)

method_box_right = widgets.Dropdown(
    description='Algorithm:   ',
    value='ground-truth',
    options=test_df.method.unique().tolist(),
)

container = widgets.HBox(children=[method_box_left, method_box_right])

traces = plot_graph(G, return_traces=True)

fig = make_subplots(
        rows=1, cols=2, subplot_titles=['Method 1', 'Method 2'],
        column_widths=[0.5, 0.5],
        vertical_spacing=0.0,)

fig.update_layout(title=dict(text=f'Clustering visualization:   {G.name}    |V| = {G.order():,}   |E| = {G.size():,}'),
                  width=1500, height=700,
                  showlegend=False)

for col in [1, 2]:
    fig.update_xaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    fig.update_yaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    
g = go.FigureWidget(fig)
g.add_traces(traces, rows=1, cols=1)  # initialize both panels with the same graph
g.add_traces(traces, rows=1, cols=2);

In [105]:
method_box_left.observe(response_both, names="value")
method_box_right.observe(response_both, names="value")

# run this
widgets.VBox([container, g])

# add slider for tau